In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from preclean import preprocessing

In [2]:
# ── Chargement train ─────────────────────────────────
df_transac = pd.read_csv('data/train_transaction.csv')
df_id      = pd.read_csv('data/train_identity.csv')
df_train   = preprocessing(df_transac, df_id)

X = df_train.drop(columns=['TransactionID', 'isFraud'])
y = df_train['isFraud']

# ── Chargement test Kaggle ───────────────────────────
df_test_t  = pd.read_csv('data/test_transaction.csv')
df_test_i  = pd.read_csv('data/test_identity.csv')
df_test    = preprocessing(df_test_t, df_test_i)

X_test = df_test.drop(columns=['TransactionID'])

# Aligner les colonnes (get_dummies peut créer des écarts)
X_test = X_test.reindex(columns=X.columns, fill_value=0)

✅ Shape finale : (590540, 798)
   NaN résiduels (float, gérés par XGBoost) : 267300666
✅ Shape finale : (506691, 446)
   NaN résiduels (float, gérés par XGBoost) : 87321159


In [3]:
# ── Paramètres ───────────────────────────────────────
params = dict(
    scale_pos_weight      = len(y[y==0]) / len(y[y==1]),
    n_estimators          = 1000,
    max_depth             = 6,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 5,
    tree_method           = 'hist',
    eval_metric           = 'auc',
    early_stopping_rounds = 50,
    random_state          = 42,
    n_jobs                = -1
)

In [4]:
# Filet de sécurité : supprimer toute colonne str restante
str_cols = X.select_dtypes(include='object').columns.tolist()
if str_cols:
    print(f"⚠️ Colonnes str supprimées : {str_cols}")
    X = X.drop(columns=str_cols)
    X_test = X_test.drop(columns=str_cols, errors='ignore')

/var/folders/tm/ltl1_5rx0lq1ps9g4gbyst380000gn/T/ipykernel_57308/2466497679.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = X.select_dtypes(include='object').columns.tolist()


⚠️ Colonnes str supprimées : ['P_emaildomain', 'R_emaildomain', 'id_16', 'id_35', 'os_family', 'browser_family', 'DeviceInfo_brand']


In [ ]:
# ── Validation croisée sur le TRAIN uniquement ───────
# (pour savoir si ton modèle est bon avant de soumettre)
skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof    = np.zeros(len(X))
scores = []

for fold, (idx_train, idx_val) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[idx_train], X.iloc[idx_val]
    y_tr, y_val = y.iloc[idx_train], y.iloc[idx_val]

    model = XGBClassifier(**params)

    str_cols = X.select_dtypes(include='object').columns.tolist()
    print(f"{len(str_cols)} colonnes str restantes : {str_cols}")

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=100
    )

    oof[idx_val] = model.predict_proba(X_val)[:, 1]
    scores.append(roc_auc_score(y_val, oof[idx_val]))
    print(f"Fold {fold+1} AUC : {scores[-1]:.4f}")

print(f"\nAUC moyenne    : {np.mean(scores):.4f}")
print(f"AUC OOF totale : {roc_auc_score(y, oof):.4f}")

0 colonnes str restantes : []
[0]	validation_0-auc:0.84542
[100]	validation_0-auc:0.91224


In [ ]:
# ── Réentraînement sur TOUT le train ─────────────────
# early_stopping nécessite un eval_set, on prend 5% du train
from sklearn.model_selection import train_test_split
X_main, X_watch, y_main, y_watch = train_test_split(
    X, y, test_size=0.05, stratify=y, random_state=42
)

model_final = XGBClassifier(**params)
model_final.fit(
    X_main, y_main,
    eval_set=[(X_watch, y_watch)],
    verbose=False
)

In [ ]:
# ── Soumission ───────────────────────────────────────
preds = model_final.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'TransactionID' : df_test['TransactionID'],
    'isFraud'       : preds
})
submission.to_csv('submission.csv', index=False)
print("submission.csv créé ✓")